## Applying LoRA to a Hugging Face Transformer Model

In [15]:
!pip install transformers datasets accelerate --quiet

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
import math

In [17]:
class LoRALinear(nn.Module):
  def __init__(self,
              in_features: int,
              out_features: int,
              rank: int = 8,
              alpha: float = 16.0,
              dropout: float = 0.0,
              bias: bool = True,
              merge_weights: bool = False,   # set True at inference to fold adapter into W
            ):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.rank = rank
    self.alpha = alpha
    self.scaling = alpha / rank
    self.merge_weights = merge_weights

    # Original frozen weight
    self.weight = nn.Parameter(torch.empty((out_features, in_features)))
    if bias:
      self.bias = nn.Parameter(torch.empty(out_features))
    else:
      self.register_parameter('bias', None)

    # LoRA Matrices
    self.lora_A = nn.Parameter(torch.empty((rank, in_features)))
    nn.init.normal_(self.lora_A, mean=0, std=1)
    self.lora_B = nn.Parameter(torch.empty((out_features, rank)))
    nn.init.normal_(self.lora_B, mean=0, std=1)

    # Dropout
    self.dropout = nn.Dropout(dropout)

    # Track whether adapter is merged
    self.is_merged = False

    # Initialize weights
    self.reset_parameters()

  def reset_parameters(self):
    # Standard Linear init for the base weight
    nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
    if self.bias is not None:
      fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
      bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
      nn.init.uniform_(self.bias, -bound, bound)

    # LoRA init (common practice)
      nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
      nn.init.zeros_(self.lora_B) # so ΔW starts at 0

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    if self.is_merged:
      return F.linear(x, self.weight, self.bias)

    result = F.linear(x, self.weight, self.bias)

    lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
    result = result + lora_out * self.scaling
    return result

  def merge(self):
      """Fold LoRA weights into the base weight (useful for inference)."""
      if self.merged:
          return
      self.weight.data += (self.lora_B @ self.lora_A) * self.scaling
      self.merged = True

  def unmerge(self):
    """Undo the merge (rarely needed)."""
    if not self.merged:
        return
    self.weight.data -= (self.lora_B @ self.lora_A) * self.scaling
    self.merged = False

In [18]:
def apply_lora(
    model: nn.Module,
    target_modules: list[str] = None,
    rank: int = 8,
    alpha: float = 16.0,
    dropout: float = 0.0
    ):
  """
    Recursively replace Linear layers whose names contain any of the
    target_modules strings with LoRALinear.
  """
  if target_modules is None:
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

  for name, module in model.named_children():
    if isinstance(module, nn.Linear) and any(t in name for t in target_modules):
      # Create LoRA version
      lora_layer = LoRALinear(
          in_features=module.in_features,
          out_features=module.out_features,
          rank=rank,
          alpha=alpha,
          dropout=dropout,
          bias=module.bias is not None,
          merge_weights=False
      )

      # Copy original weights
      lora_layer.weight.data = module.weight.data.clone()
      if module.bias is not None:
        lora_layer.bias.data = module.bias.data.clone()

      # Freeze original weight
      module.weight.requires_grad = False
      if lora_layer.bias is not None:
        lora_layer.bias.requires_grad = False

      # Replace with LoRA
      setattr(model, name, lora_layer)
    else:
      # Recurse
      apply_lora(module, target_modules, rank, alpha, dropout)

  return model

In [19]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"
model_hf = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("\nHugging Face Model structure BEFORE applying LoRA:")
print(model_hf)

target_lora_modules_hf = [
    "q_proj", "k_proj", "v_proj", "out_proj",
    "lin1", "lin2" # For the feed-forward network
]

apply_lora(
    model_hf,
    target_modules=target_lora_modules_hf,
    rank=8,
    alpha=16.0,
    dropout=0.1
)

print("\nHugging Face Model structure AFTER applying LoRA:")
print(model_hf)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Hugging Face Model structure BEFORE applying LoRA:
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
 

In [20]:
lora_applied_count_hf = 0
for name, module in model_hf.named_modules():
    if isinstance(module, LoRALinear):
        print(f"  LoRALinear applied to: {name}")
        lora_applied_count_hf += 1

if lora_applied_count_hf > 0:
    print(f"\nSuccessfully applied LoRA to {lora_applied_count_hf} modules in the Hugging Face model.")
else:
    print("\nLoRA was not applied to any modules in the Hugging Face model. Check target_modules and model structure.")

  LoRALinear applied to: distilbert.transformer.layer.0.ffn.lin1
  LoRALinear applied to: distilbert.transformer.layer.0.ffn.lin2
  LoRALinear applied to: distilbert.transformer.layer.1.ffn.lin1
  LoRALinear applied to: distilbert.transformer.layer.1.ffn.lin2
  LoRALinear applied to: distilbert.transformer.layer.2.ffn.lin1
  LoRALinear applied to: distilbert.transformer.layer.2.ffn.lin2
  LoRALinear applied to: distilbert.transformer.layer.3.ffn.lin1
  LoRALinear applied to: distilbert.transformer.layer.3.ffn.lin2
  LoRALinear applied to: distilbert.transformer.layer.4.ffn.lin1
  LoRALinear applied to: distilbert.transformer.layer.4.ffn.lin2
  LoRALinear applied to: distilbert.transformer.layer.5.ffn.lin1
  LoRALinear applied to: distilbert.transformer.layer.5.ffn.lin2

Successfully applied LoRA to 12 modules in the Hugging Face model.
